# W05 — Model: Ranking Signal Analysis

**Lane:** Ranking Signal Analysis
**Month used:** `month=2026-03` (mid-panel, not the sealed final month)
**Compares against:** the Week-4 baseline (`baseline_action_score`, rule: staleness + CTR-gap,
reason code `stale_underperforming_ctr`)

> **Before you submit:** run every cell top-to-bottom in Colab with `HF_TOKEN` set as a Secret.
> All modeling logic is complete and ready to execute; the actual metrics and feature-importance
> numbers can only come from running it against the real gated warehouse data. Run All, read the
> model-vs-baseline table, fill in the interpretation lines flagged `# FILL AFTER RUN`, then commit.


## Setup

In [ ]:
import duckdb, os, json
import numpy as np
import pandas as pd
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"SET hf_token='{os.environ['HF_TOKEN']}';")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

import pathlib
pathlib.Path("work/outputs").mkdir(parents=True, exist_ok=True)


## 1) Method Choice And Why

**Models used:** Logistic Regression (primary) and Random Forest (comparison), both from this
week's menu.

**Why these two, for this lane:**
- Ranking Signal Analysis asks *which safe signals are associated with visibility/movement, and
  how strongly* — that calls for a model whose coefficients or importances can be read and
  explained, not just a high score. Logistic Regression gives directly interpretable coefficients
  (effect size and direction per signal); permutation importance on top of it and on the Random
  Forest gives a second, model-agnostic read on which features actually matter.
- Random Forest is included only as a comparison point to check whether nonlinearity/interactions
  buy anything over the linear model — not because more complexity is assumed to be better
  (the task explicitly says complexity alone isn't rewarded). If it doesn't clearly beat Logistic
  Regression on the same metric, Logistic Regression stays the pick, since it's the one whose
  "why" is easiest to defend to a reviewer.
- Gradient Boosting and clustering are on the menu but aren't the right fit here: boosting adds
  complexity this lane's five safe features don't need to earn, and clustering answers a different
  question (archetypes, not signal-vs-outcome association) — that's Lane 3's job.

**Label used (same proxy as Weeks 3–4, kept consistent on purpose):**
`label = trend_direction == 'down'`, read at the end of the March window. This is the same
beginner proxy flagged in `w03`/`w04` — a within-window bucket, not a future outcome — kept
identical here so the baseline and the model are compared on the exact same target, not two
different questions.


## 2) Split Design

**Design:** client-grouped train/test split (`GroupShuffleSplit` on `client_hash_id`), not a
plain random split.

**Why:** pages from the same client can share patterns (site-wide staleness habits, a shared CMS,
similar content templates) that a plain random split could let leak between train and test —
the model could partly memorize a client instead of learning a generalizable signal (section 12
of the lane guide: client/group holdout when rows from the same group may share patterns). Holding
out whole clients means the reported metrics reflect performance on clients the model has not
effectively already seen.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

features_df = con.sql(f"""
WITH monthly AS (
  SELECT
    f.content_hash_id,
    f.client_hash_id,
    SUM(f.impressions) AS impressions_30d,
    SUM(f.clicks)       AS clicks_30d,
    AVG(f.position)     AS avg_position_30d,
    d.word_count         AS word_count,
    d.days_since_last_update AS days_since_last_update
  FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') f
  JOIN read_parquet('{BASE}/dim_content/*.parquet') d
    ON f.content_hash_id = d.content_hash_id
  GROUP BY 1, 2, d.word_count, d.days_since_last_update
),
latest_trend AS (
  SELECT content_hash_id, client_hash_id, trend_direction
  FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY content_hash_id, client_hash_id ORDER BY report_date DESC
  ) = 1
)
SELECT m.*, t.trend_direction
FROM monthly m
JOIN latest_trend t USING (content_hash_id, client_hash_id)
""").df()

features_df["ctr_30d"] = features_df["clicks_30d"] / features_df["impressions_30d"].replace(0, np.nan)
features_df["ctr_30d"] = features_df["ctr_30d"].fillna(0)
features_df["label"] = (features_df["trend_direction"] == "down").astype(int)

FEATURE_COLS = ["impressions_30d", "clicks_30d", "ctr_30d", "avg_position_30d", "word_count"]
X = features_df[FEATURE_COLS].fillna(0)
y = features_df["label"]
groups = features_df["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])
assert train_clients.isdisjoint(test_clients), "client leakage across split!"

print("train rows:", len(X_train), " test rows:", len(X_test))
print("train clients:", len(train_clients), " test clients:", len(test_clients))


## 3) Train + Compare Vs My Baseline

The Week-4 baseline rule (`baseline_action_score`) is recomputed on the same test rows so it is
scored on the exact same split and the exact same label as the models — a rule needs no fitting,
only evaluating.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)[:k]
    return y_true.values[order].mean()

# --- baseline: Week-4 rule, evaluated on the test split only ---
test_df = features_df.iloc[test_idx].copy()
tier = pd.cut(
    test_df["avg_position_30d"], bins=[-1, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"]
)
tier_median_ctr = test_df.groupby(tier)["ctr_30d"].transform("median")
staleness_norm = (test_df["days_since_last_update"] / 365).clip(0, 1)
ctr_gap_norm = ((tier_median_ctr - test_df["ctr_30d"]) / tier_median_ctr.replace(0, np.nan)).clip(0, 1).fillna(0)
baseline_score_test = 0.5 * staleness_norm + 0.5 * ctr_gap_norm

# --- Logistic Regression ---
logreg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
logreg_score = logreg.predict_proba(X_test)[:, 1]

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_score = rf.predict_proba(X_test)[:, 1]

results = pd.DataFrame([
    {
        "method": "baseline (Week 4 rule)",
        "roc_auc": roc_auc_score(y_test, baseline_score_test),
        "average_precision": average_precision_score(y_test, baseline_score_test),
        "precision_at_50": precision_at_k(y_test, baseline_score_test.values, 50),
    },
    {
        "method": "logistic_regression",
        "roc_auc": roc_auc_score(y_test, logreg_score),
        "average_precision": average_precision_score(y_test, logreg_score),
        "precision_at_50": precision_at_k(y_test, logreg_score, 50),
    },
    {
        "method": "random_forest",
        "roc_auc": roc_auc_score(y_test, rf_score),
        "average_precision": average_precision_score(y_test, rf_score),
        "precision_at_50": precision_at_k(y_test, rf_score, 50),
    },
])
results


**Read the table above before writing anything else.** `# FILL AFTER RUN` — one or two
sentences: which method wins on `precision_at_50` (the metric that matches "a reviewer checks the
top 50"), and whether Random Forest's improvement (if any) over Logistic Regression is big enough
to justify losing the linear model's easy interpretability. If Random Forest doesn't clearly win,
the write-up should say Logistic Regression is the kept model — complexity alone isn't a reason to
prefer it.


In [ ]:
with open("work/outputs/w05_model_metrics.json", "w") as f:
    json.dump({
        "lane": "ranking_signal_analysis",
        "month": MONTH,
        "label": "trend_direction == 'down' (within-window proxy, same as w03/w04)",
        "split": "GroupShuffleSplit on client_hash_id, test_size=0.25, random_state=42",
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        "results": results.to_dict(orient="records"),
    }, f, indent=2)
print("wrote work/outputs/w05_model_metrics.json")


## 4) Errors And Interpretation

**Logistic Regression coefficients** (direction and rough size of each signal's effect on the
proxy label):


In [ ]:
coef_table = pd.DataFrame({
    "feature": FEATURE_COLS,
    "coefficient": logreg.coef_[0],
}).sort_values("coefficient", key=abs, ascending=False)
coef_table


**Permutation importance** (model-agnostic — how much each feature's shuffling hurts test
performance, cross-checked against the coefficients above):


In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    logreg, X_test, y_test, n_repeats=20, random_state=42, scoring="roc_auc"
)
perm_table = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)
perm_table


**Where the model disagrees with the baseline** — rows the two rank very differently are the
most useful place to look for what each one is missing:


In [ ]:
test_df = test_df.reset_index(drop=True)
disagreement = pd.DataFrame({
    "content_hash_id": test_df["content_hash_id"],
    "client_hash_id": test_df["client_hash_id"],
    "label": y_test.values,
    "baseline_score": baseline_score_test.values,
    "model_score": logreg_score,
})
disagreement["baseline_rank"] = disagreement["baseline_score"].rank(ascending=False)
disagreement["model_rank"] = disagreement["model_score"].rank(ascending=False)
disagreement["rank_gap"] = (disagreement["baseline_rank"] - disagreement["model_rank"]).abs()

top_disagreements = disagreement.sort_values("rank_gap", ascending=False).head(10)
top_disagreements


**Error interpretation:** `# FILL AFTER RUN` — a few sentences once the tables above are
real:
- Which feature(s) dominate the coefficient table and the permutation importance — do the two
  agree? If they don't, say so plainly rather than picking whichever story is more convincing.
- For the top disagreement rows: eyeball a few — is the model catching something the fixed rule's
  two signals miss (e.g. impressions/clicks trend the rule doesn't see directly), or is it
  latching onto noise in a low-volume tier? Note which.
- Whether the false positives/negatives cluster in any obvious way (e.g. a specific position tier
  or a small set of clients) — that's a signal the label or the split needs more work, not
  necessarily that the model is bad.


## 5) Self-Check

- [ ] Model fits the lane (interpretable + a comparison model, matching Ranking Signal Analysis's
      need to explain association strength) — section 1.
- [ ] Method choice explained, including why Gradient Boosting/clustering were not the right fit
      here — section 1.
- [ ] Valid split/validation design: client-grouped, with an explicit disjoint-client assertion —
      section 2.
- [ ] Compared against the Week-4 baseline on the exact same test split and the exact same label —
      section 3.
- [ ] Useful metrics reported: ROC AUC, average precision, and precision@50 (matches "reviewer
      checks the top 50" from the lane guide) — section 3.
- [ ] Features/errors interpreted: coefficients, permutation importance, and a baseline-vs-model
      disagreement look — section 4.
- [ ] Does not reward complexity alone: Random Forest is kept only if it clearly beats Logistic
      Regression on `precision_at_50` — decision written out in section 3.
- [ ] Read this week's research paper (tracked outside this notebook, per the card).
